## validating calibration

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import yaml
from ewatercycle.observation.grdc import get_grdc_data
from IPython.display import display


In [2]:
PROJECT_ROOT = Path("/home/avandervee3/aral_sea_full_project")
CONFIG_PATH = PROJECT_ROOT / "config_aral.yaml"
HISTORICAL_PATH = PROJECT_ROOT / "results/runs/pcrglobwb/stations/final/historical.nc"
GRDC_DATA_HOME = PROJECT_ROOT / "data/grdc/Daily"
OUTPUT_PATH = PROJECT_ROOT / "outputs/validation_top15_kge.csv"

WINDOWS = [
    ("1950-01-01", "1954-12-31"),
    ("1955-01-01", "1959-12-31"),
]
RUN_IDS = ["r001", "r002", "r003", "r004"]
TOP_STATION_COUNT = 50

with open(CONFIG_PATH, encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

stations_df = (
    pd.DataFrame(config["discharge_stations"])
    .sort_values("weight", ascending=False)
    .head(TOP_STATION_COUNT)
    .reset_index(drop=True)
)

stations_df[["name", "station_key", "grdc_id", "weight"]]


,name,station_key,grdc_id,weight
0,Chatly,Chatly,2817100,670.820393
1,Kerki,Kerki,2617110,555.877684
2,Tyumen-aryk,Tyumen-aryk,2316200,467.974358
3,Bekabad,Bekabad,2816203,376.828874
4,Kal,Kal,2416202,300.000000
5,Uch-kurgan,Uch-kurgan,2416850,241.660919
6,Kazalinsk,Kazalinsk,2316201,230.000000
7,Karaozek,Karaozek,2316204,230.000000
8,Ust. Kekirim,Ust. Kekirim,2416860,186.010752
9,Tutkaul,Tutkaul,2517900,176.635217


In [3]:
historical_data = xr.open_dataset(HISTORICAL_PATH)

historical_data


<xarray.Dataset> Size: 71MB
Dimensions:     (time: 29586, station: 50, experiment: 12)
Coordinates:
  * time        (time) datetime64[ns] 237kB 1940-01-01 1940-01-02 ... 2020-12-31
    lat         (station) float32 200B ...
    lon         (station) float32 200B ...
  * experiment  (experiment) <U124 6kB 'run_id=r001__model=era5__scenario=era...
  * station     (station) <U14 3kB 'Tyumen-aryk' 'Kazalinsk' ... 'Obizarang'
Data variables:
    discharge   (experiment, time, station) float32 71MB ...
Attributes: (12/19)
    title:                     Aggregated PCR-GLOBWB station discharge (histo...
    summary:                   Station discharge aggregated across experiment...
    era:                       historical
    station_count:             50
    station_names:             Tyumen-aryk,Kazalinsk,Karaozek,Kal,Dazgon,Anda...
    station_latitudes:         43.950000,45.700000,44.960000,40.875000,39.792...
    ...                        ...
    source_parameter_sets:     uncalibrated,overall_calibrated,amu_calibrated...
    source_job_ids:            r001_000,r001_001,r001_002,r002_000,r002_001,r...
    config_yaml:               config_aral.yaml
    created_at:                2026-06-01T00:42:41+00:00
    processing_step:           aggregate_pcrglobwb_station_outputs
    merge_axis:                experiment

In [4]:
def calculate_kge(observed, simulated):
    observed = np.asarray(observed, dtype=float)
    simulated = np.asarray(simulated, dtype=float)
    mask = ~(np.isnan(observed) | np.isnan(simulated))
    obs = observed[mask]
    sim = simulated[mask]

    if len(obs) < 2:
        return np.nan

    obs_std = np.std(obs)
    obs_mean = np.mean(obs)
    if obs_std == 0 or obs_mean == 0:
        return np.nan

    sim_std = np.std(sim)
    sim_mean = np.mean(sim)
    correlation = np.corrcoef(obs, sim)[0, 1]
    alpha = sim_std / obs_std
    beta = sim_mean / obs_mean
    return 1 - np.sqrt((correlation - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)


def normalize_series(series):
    series = series.copy()
    series.index = pd.to_datetime(series.index).tz_localize(None)
    return series.sort_index()


def resolve_experiment_label(run_id):
    labels = [str(value) for value in historical_data["experiment"].values]
    direct = [label for label in labels if label == run_id or label.startswith(f"run_id={run_id}")]
    if direct:
        return direct[0]
    partial = [label for label in labels if run_id in label]
    if partial:
        return partial[0]
    raise KeyError(f"Could not find experiment label for {run_id}")


def get_simulated_series(run_id, station_name):
    experiment_label = resolve_experiment_label(run_id)
    discharge = historical_data["discharge"].sel(experiment=experiment_label, station=station_name)
    return normalize_series(discharge.to_series())


def get_grdc_series(station_id, start_date, end_date):
    observations = get_grdc_data(
        station_id=int(station_id),
        start_time=f"{start_date}T00:00Z",
        end_time=f"{end_date}T00:00Z",
        data_home=str(GRDC_DATA_HOME),
    )
    series = observations["streamflow"].to_series()
    return normalize_series(series)


def compute_kge_row(run_id, station_row, start_date, end_date):
    simulated = get_simulated_series(run_id, station_row["name"])
    observed = get_grdc_series(station_row["grdc_id"], start_date, end_date)
    combined = pd.concat(
        [simulated.rename("simulated"), observed.rename("observed")],
        axis=1,
        join="inner",
    ).dropna()

    return {
        "run_id": run_id,
        "experiment": resolve_experiment_label(run_id),
        "station_name": station_row["name"],
        "station_key": station_row.get("station_key", station_row["name"]),
        "grdc_id": int(station_row["grdc_id"]),
        "weight": float(station_row["weight"]),
        "window_start": start_date,
        "window_end": end_date,
        "n_points": int(len(combined)),
        "kge": calculate_kge(combined["observed"].values, combined["simulated"].values),
    }


In [5]:
results = []
for run_id in RUN_IDS:
    for window_start, window_end in WINDOWS:
        for _, station_row in stations_df.iterrows():
            results.append(compute_kge_row(run_id, station_row, window_start, window_end))

results_df = pd.DataFrame(results).sort_values(["run_id", "window_start", "weight"], ascending=[True, True, False]).reset_index(drop=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False)

results_df


,run_id,experiment,station_name,station_key,grdc_id,weight,window_start,window_end,n_points,kge
0,r001,run_id=r001__model=era5__scenario=era5__type=r...,Chatly,Chatly,2817100,670.820393,1950-01-01,1954-12-31,1736,-2.904523
1,r001,run_id=r001__model=era5__scenario=era5__type=r...,Kerki,Kerki,2617110,555.877684,1950-01-01,1954-12-31,932,-0.034091
2,r001,run_id=r001__model=era5__scenario=era5__type=r...,Tyumen-aryk,Tyumen-aryk,2316200,467.974358,1950-01-01,1954-12-31,1798,-7.573793
3,r001,run_id=r001__model=era5__scenario=era5__type=r...,Bekabad,Bekabad,2816203,376.828874,1950-01-01,1954-12-31,0,NaN
4,r001,run_id=r001__model=era5__scenario=era5__type=r...,Kal,Kal,2416202,300.000000,1950-01-01,1954-12-31,0,NaN
...,...,...,...,...,...,...,...,...,...,...
395,r004,run_id=r004__model=era5__scenario=era5__type=r...,Varganza,Varganza,2817310,22.605309,1955-01-01,1959-12-31,0,NaN
396,r004,run_id=r004__model=era5__scenario=era5__type=r...,Changet,Changet,2416780,19.519221,1955-01-01,1959-12-31,0,NaN
397,r004,run_id=r004__model=era5__scenario=era5__type=r...,Ust. Tostu,Ust. Tostu,2416900,19.131126,1955-01-01,1959-12-31,0,NaN
398,r004,run_id=r004__model=era5__scenario=era5__type=r...,Alibegi,Alibegi,2517470,19.026298,1955-01-01,1959-12-31,0,NaN


In [6]:
summary = {
    "runs": len(RUN_IDS),
    "stations": len(stations_df),
    "windows": len(WINDOWS),
    "rows_written": len(results_df),
    "csv_path": str(OUTPUT_PATH),
}
summary


{'runs': 4,
 'stations': 50,
 'windows': 2,
 'rows_written': 400,
 'csv_path': '/home/avandervee3/aral_sea_full_project/outputs/validation_top15_kge.csv'}

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np


def read_grdc_monthly_values(start_date, end_date, station_id, data_dir):
    """Read GRDC monthly discharge values for a station and date range (inclusive).
    
    Parameters
    ----------
    start_date : str or datetime-like
        Start date, e.g. "1966-01-01".
    end_date : str or datetime-like
        End date, e.g. "1966-12-31".
    station_id : int or str
        GRDC station id used in file name.
    data_dir : str or Path
        Directory that contains GRDC monthly text files.
    
    Returns
    -------
    pandas.DataFrame
        Columns: date, value
    """
    data_dir = Path(data_dir)
    station_id = str(station_id)

    # Support small naming differences between downloaded GRDC files.
    candidate_names = [
        f"{station_id}_Q_Month.txt",
        f"{station_id}_Q_month.txt",
    ]

    file_path = None
    for name in candidate_names:
        candidate = data_dir / name
        if candidate.exists():
            file_path = candidate
            break

    if file_path is None:
        raise FileNotFoundError(
            f"GRDC file not found in {data_dir}. Tried: {', '.join(candidate_names)}"
        )

    # GRDC text files can contain non-UTF8 symbols; latin-1 safely decodes them.
    df = pd.read_csv(
        file_path,
        sep=';',
        comment='#',
        skipinitialspace=True,
        engine='python',
        encoding='latin-1'
    )

    # Clean column names produced by GRDC header spacing.
    df.columns = [col.strip() for col in df.columns]

    if 'YYYY-MM-DD' not in df.columns:
        raise ValueError(f"Could not find 'YYYY-MM-DD' column in {file_path.name}")

    # Use Original if available and valid; otherwise use Calculated
    original_col = 'Original' if 'Original' in df.columns else None
    calculated_col = 'Calculated' if 'Calculated' in df.columns else None

    df['date'] = pd.to_datetime(df['YYYY-MM-DD'], errors='coerce', utc=True)

    # Prefer Original; merge with Calculated if Original is missing/invalid
    if original_col and calculated_col:
        df['original_val'] = pd.to_numeric(df[original_col], errors='coerce')
        df['calculated_val'] = pd.to_numeric(df[calculated_col], errors='coerce')
        # Use original if valid (not NaN and not -999), otherwise calculated
        df['value'] = df['original_val'].copy()
        mask_use_calc = (df['original_val'].isna()) | (df['original_val'] == -999.0)
        df.loc[mask_use_calc, 'value'] = df.loc[mask_use_calc, 'calculated_val']
    elif original_col:
        df['value'] = pd.to_numeric(df[original_col], errors='coerce')
    elif calculated_col:
        df['value'] = pd.to_numeric(df[calculated_col], errors='coerce')
    else:
        raise ValueError(f"Found neither 'Original' nor 'Calculated' column in {file_path.name}")

    start_ts = pd.to_datetime(start_date, utc=True)
    end_ts = pd.to_datetime(end_date, utc=True)

    out = df.loc[
        (df['date'] >= start_ts)
        & (df['date'] <= end_ts)
        & (df['value'].notna())
        & (df['value'] != -999.0),
        ['date', 'value']
    ].copy()

    out = out.sort_values('date').reset_index(drop=True)
    return out

In [8]:
GRDC_MONTHLY_HOME = PROJECT_ROOT / "data/grdc/Monthly"

def get_simulated_monthly_series(run_id, station_name):
    """Resample daily simulated discharge to monthly means."""
    daily = get_simulated_series(run_id, station_name)
    return daily.resample("MS").mean()

def get_grdc_monthly_series(station_id, start_date, end_date):
    """Read GRDC monthly values using the custom parser."""
    df = read_grdc_monthly_values(
        start_date=start_date,
        end_date=end_date,
        station_id=int(station_id),
        data_dir=GRDC_MONTHLY_HOME,
    )
    series = df.set_index("date")["value"]
    series.index = pd.to_datetime(series.index).tz_localize(None)
    return series.sort_index()

def compute_kge_row(run_id, station_row, start_date, end_date):
    # --- daily ---
    sim_daily = get_simulated_series(run_id, station_row["name"])
    obs_daily = get_grdc_series(station_row["grdc_id"], start_date, end_date)
    combined_daily = pd.concat(
        [sim_daily.rename("simulated"), obs_daily.rename("observed")],
        axis=1, join="inner",
    ).dropna()

    # --- monthly ---
    sim_monthly = get_simulated_monthly_series(run_id, station_row["name"])
    try:
        obs_monthly = get_grdc_monthly_series(station_row["grdc_id"], start_date, end_date)
        combined_monthly = pd.concat(
            [sim_monthly.rename("simulated"), obs_monthly.rename("observed")],
            axis=1, join="inner",
        ).dropna()
        kge_monthly = calculate_kge(
            combined_monthly["observed"].values,
            combined_monthly["simulated"].values,
        )
        n_monthly = len(combined_monthly)
    except FileNotFoundError:
        kge_monthly = np.nan
        n_monthly = 0

    return {
        "run_id": run_id,
        "experiment": resolve_experiment_label(run_id),
        "station_name": station_row["name"],
        "station_key": station_row.get("station_key", station_row["name"]),
        "grdc_id": int(station_row["grdc_id"]),
        "weight": float(station_row["weight"]),
        "window_start": start_date,
        "window_end": end_date,
        "n_points": int(len(combined_daily)),
        "kge": calculate_kge(
            combined_daily["observed"].values,
            combined_daily["simulated"].values,
        ),
        "n_points_monthly": n_monthly,
        "kge_monthly": kge_monthly,
    }

In [9]:
results = []
for run_id in RUN_IDS:
    for window_start, window_end in WINDOWS:
        for _, station_row in stations_df.iterrows():
            results.append(compute_kge_row(run_id, station_row, window_start, window_end))

results_df = pd.DataFrame(results).sort_values(["run_id", "window_start", "weight"], ascending=[True, True, False]).reset_index(drop=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_PATH, index=False)

results_df


,run_id,experiment,station_name,station_key,grdc_id,weight,window_start,window_end,n_points,kge,n_points_monthly,kge_monthly
0,r001,run_id=r001__model=era5__scenario=era5__type=r...,Chatly,Chatly,2817100,670.820393,1950-01-01,1954-12-31,1736,-2.904523,57,-2.944425
1,r001,run_id=r001__model=era5__scenario=era5__type=r...,Kerki,Kerki,2617110,555.877684,1950-01-01,1954-12-31,932,-0.034091,30,-0.024218
2,r001,run_id=r001__model=era5__scenario=era5__type=r...,Tyumen-aryk,Tyumen-aryk,2316200,467.974358,1950-01-01,1954-12-31,1798,-7.573793,60,-7.699270
3,r001,run_id=r001__model=era5__scenario=era5__type=r...,Bekabad,Bekabad,2816203,376.828874,1950-01-01,1954-12-31,0,NaN,60,-6.431098
4,r001,run_id=r001__model=era5__scenario=era5__type=r...,Kal,Kal,2416202,300.000000,1950-01-01,1954-12-31,0,NaN,60,-2.729699
...,...,...,...,...,...,...,...,...,...,...,...,...
395,r004,run_id=r004__model=era5__scenario=era5__type=r...,Varganza,Varganza,2817310,22.605309,1955-01-01,1959-12-31,0,NaN,60,0.088150
396,r004,run_id=r004__model=era5__scenario=era5__type=r...,Changet,Changet,2416780,19.519221,1955-01-01,1959-12-31,0,NaN,60,-0.511095
397,r004,run_id=r004__model=era5__scenario=era5__type=r...,Ust. Tostu,Ust. Tostu,2416900,19.131126,1955-01-01,1959-12-31,0,NaN,60,-0.400372
398,r004,run_id=r004__model=era5__scenario=era5__type=r...,Alibegi,Alibegi,2517470,19.026298,1955-01-01,1959-12-31,0,NaN,60,0.575535


In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

def df_to_latex_kge(
    csv_path: str | Path,
    output_path: str | Path,
    caption: str = "Daily and monthly KGE scores for calibration and validation periods across parameter sets. Dashes indicate no daily observations available for that station.",
    label: str = "tab:kge_validation_appendix",
):
    df = pd.read_csv(csv_path)

    def parse_param_set(experiment):
        for part in str(experiment).split("__"):
            if part.startswith("parameter_set="):
                return part.replace("parameter_set=", "")
        return experiment

    df["param_set"] = df["experiment"].apply(parse_param_set)

    param_labels = {
        "uncalibrated":       "Uncalibrated",
        "overall_calibrated": "Overall cal.",
        "amu_calibrated":     "Amu Darya cal.",
        "syr_calibrated":     "Syr Darya cal.",
    }

    param_sets = list(dict.fromkeys(df["param_set"].tolist()))

    period_labels = {}
    sorted_windows = sorted(df["window_start"].unique())
    period_labels[sorted_windows[0]] = "Calibration"
    for w in sorted_windows[1:]:
        period_labels[w] = "Validation"

    station_order = (
        df[["station_name", "weight"]]
        .drop_duplicates()
        .sort_values("weight", ascending=False)["station_name"]
        .tolist()
    )

    def fmt(val):
        if pd.isna(val):
            return "---"
        return f"${val:.2f}$"

    n_params = len(param_sets)
    col_spec = "ll" + "r" * n_params + "r" * n_params
    col_headers = " & ".join(param_labels.get(p, p) for p in param_sets)
    total_cols = 2 + 2 * n_params

    lines = []
    lines += [
        r"\begin{longtable}{" + col_spec + "}",
        r"\caption{" + caption + r"} \label{" + label + r"} \\",
        r"\toprule",
        r" & & \multicolumn{" + str(n_params) + r"}{c}{KGE daily (--)} & \multicolumn{" + str(n_params) + r"}{c}{KGE monthly (--)} \\",
        r"\cmidrule(lr){3-" + str(2 + n_params) + r"} \cmidrule(lr){" + str(3 + n_params) + r"-" + str(total_cols) + r"}",
        r"Station & Period & " + col_headers + " & " + col_headers + r" \\",
        r"\midrule",
        r"\endfirsthead",
        r"\multicolumn{" + str(total_cols) + r"}{l}{\textit{...continued from previous page}} \\",
        r"\toprule",
        r" & & \multicolumn{" + str(n_params) + r"}{c}{KGE daily (--)} & \multicolumn{" + str(n_params) + r"}{c}{KGE monthly (--)} \\",
        r"\cmidrule(lr){3-" + str(2 + n_params) + r"} \cmidrule(lr){" + str(3 + n_params) + r"-" + str(total_cols) + r"}",
        r"Station & Period & " + col_headers + " & " + col_headers + r" \\",
        r"\midrule",
        r"\endhead",
        r"\midrule",
        r"\multicolumn{" + str(total_cols) + r"}{r}{\textit{Continued on next page...}} \\",
        r"\endfoot",
        r"\bottomrule",
        r"\endlastfoot",
    ]

    for station in station_order:
        station_df = df[df["station_name"] == station]
        periods = sorted(station_df["window_start"].unique())
        n_periods = len(periods)
        escaped_station = station.replace(".", r".\ ")

        for i, window_start in enumerate(periods):
            period_label = period_labels.get(window_start, window_start)
            row_df = station_df[station_df["window_start"] == window_start]

            daily_vals = []
            monthly_vals = []
            for p in param_sets:
                prow = row_df[row_df["param_set"] == p]
                if prow.empty:
                    daily_vals.append("---")
                    monthly_vals.append("---")
                else:
                    daily_vals.append(fmt(prow["kge"].values[0]))
                    monthly_vals.append(fmt(prow["kge_monthly"].values[0]))

            station_cell = (
                r"\multirow{" + str(n_periods) + r"}{*}{" + escaped_station + "}"
                if i == 0 else ""
            )

            row = " & ".join(
                [station_cell, period_label] + daily_vals + monthly_vals
            ) + r" \\"
            lines.append(row)

        lines.append(r"\addlinespace")

    lines += [
        r"\end{longtable}",
    ]

    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"Written to {output_path}")


df_to_latex_kge(
    csv_path=OUTPUT_PATH,
    output_path=PROJECT_ROOT / "outputs/validation_kge_table.tex",
)

Written to /home/avandervee3/aral_sea_full_project/outputs/validation_kge_table.tex


In [13]:
def df_to_latex_kge_monthly(
    csv_path: str | Path,
    output_path: str | Path,
    caption: str = "Monthly KGE scores for calibration and validation periods across parameter sets. ",
    label: str = "tab:kge_validation_appendix",
):
    df = pd.read_csv(csv_path)

    def parse_param_set(experiment):
        for part in str(experiment).split("__"):
            if part.startswith("parameter_set="):
                return part.replace("parameter_set=", "")
        return experiment

    df["param_set"] = df["experiment"].apply(parse_param_set)

    param_labels = {
        "uncalibrated":       "Uncalibrated",
        "overall_calibrated": "Overall cal.",
        "amu_calibrated":     "Amu Darya cal.",
        "syr_calibrated":     "Syr Darya cal.",
    }

    param_sets = list(dict.fromkeys(df["param_set"].tolist()))

    period_labels = {}
    sorted_windows = sorted(df["window_start"].unique())
    period_labels[sorted_windows[0]] = "Calibration"
    for w in sorted_windows[1:]:
        period_labels[w] = "Validation"

    station_order = (
        df[["station_name", "weight"]]
        .drop_duplicates()
        .sort_values("weight", ascending=False)["station_name"]
        .tolist()
    )

    def fmt(val):
        if pd.isna(val):
            return "---"
        return f"${val:.2f}$"

    n_params = len(param_sets)
    col_spec = "ll" + "r" * n_params
    col_headers = " & ".join(param_labels.get(p, p) for p in param_sets)
    total_cols = 2 + n_params

    lines = []
    lines += [
        r"\begin{longtable}{" + col_spec + "}",
        r"\caption{" + caption + r"} \label{" + label + r"} \\",
        r"\toprule",
        r"Station & Period & " + col_headers + r" \\",
        r"\midrule",
        r"\endfirsthead",
        r"\multicolumn{" + str(total_cols) + r"}{l}{\textit{...continued from previous page}} \\",
        r"\toprule",
        r"Station & Period & " + col_headers + r" \\",
        r"\midrule",
        r"\endhead",
        r"\midrule",
        r"\multicolumn{" + str(total_cols) + r"}{r}{\textit{Continued on next page...}} \\",
        r"\endfoot",
        r"\bottomrule",
        r"\endlastfoot",
    ]

    for station in station_order:
        station_df = df[df["station_name"] == station]
        periods = sorted(station_df["window_start"].unique())
        n_periods = len(periods)
        escaped_station = station.replace(".", r".\ ")

        for i, window_start in enumerate(periods):
            period_label = period_labels.get(window_start, window_start)
            row_df = station_df[station_df["window_start"] == window_start]

            monthly_vals = []
            for p in param_sets:
                prow = row_df[row_df["param_set"] == p]
                if prow.empty:
                    monthly_vals.append("---")
                else:
                    monthly_vals.append(fmt(prow["kge_monthly"].values[0]))

            station_cell = (
                r"\multirow{" + str(n_periods) + r"}{*}{" + escaped_station + "}"
                if i == 0 else ""
            )

            row = " & ".join(
                [station_cell, period_label] + monthly_vals
            ) + r" \\"
            lines.append(row)

        lines.append(r"\addlinespace")

    lines += [
        r"\end{longtable}",
    ]

    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"Written to {output_path}")


df_to_latex_kge_monthly(
    csv_path=OUTPUT_PATH,
    output_path=PROJECT_ROOT / "outputs/validation_kge_table_monthly.tex",
)

Written to /home/avandervee3/aral_sea_full_project/outputs/validation_kge_table_monthly.tex


In [15]:
def df_to_latex_kge_monthly(
    csv_path: str | Path,
    output_path: str | Path,
    caption: str = "Monthly KGE scores for calibration and validation periods across parameter sets.",
    label: str = "tab:kge_validation_appendix",
):
    df = pd.read_csv(csv_path)

    def parse_param_set(experiment):
        for part in str(experiment).split("__"):
            if part.startswith("parameter_set="):
                return part.replace("parameter_set=", "")
        return experiment

    df["param_set"] = df["experiment"].apply(parse_param_set)

    param_labels = {
        "uncalibrated":       "Uncalibrated",
        "overall_calibrated": "Overall cal.",
        "amu_calibrated":     "Amu Darya cal.",
        "syr_calibrated":     "Syr Darya cal.",
    }

    param_sets = list(dict.fromkeys(df["param_set"].tolist()))

    period_labels = {}
    sorted_windows = sorted(df["window_start"].unique())
    period_labels[sorted_windows[0]] = "Calibration"
    for w in sorted_windows[1:]:
        period_labels[w] = "Validation"

    station_order = (
        df[["station_name", "weight"]]
        .drop_duplicates()
        .sort_values("weight", ascending=False)["station_name"]
        .tolist()
    )

    def fmt(val):
        if pd.isna(val):
            return "---"
        return f"${val:.2f}$"

    n_params = len(param_sets)
    col_spec = "ll" + "r" * n_params
    col_headers = " & ".join(param_labels.get(p, p) for p in param_sets)
    total_cols = 2 + n_params

    lines = []
    lines += [
        r"\begin{longtable}{" + col_spec + "}",
        r"\caption{" + caption + r"} \label{" + label + r"} \\",
        r"\toprule",
        r"Station & Period & " + col_headers + r" \\",
        r"\midrule",
        r"\endfirsthead",
        r"\multicolumn{" + str(total_cols) + r"}{l}{\textit{...continued from previous page}} \\",
        r"\toprule",
        r"Station & Period & " + col_headers + r" \\",
        r"\midrule",
        r"\endhead",
        r"\midrule",
        r"\multicolumn{" + str(total_cols) + r"}{r}{\textit{Continued on next page...}} \\",
        r"\endfoot",
        r"\bottomrule",
        r"\endlastfoot",
    ]

    for station in station_order:
        station_df = df[df["station_name"] == station]
        periods = sorted(station_df["window_start"].unique())
        escaped_station = station.replace(".", r".\ ")

        for i, window_start in enumerate(periods):
            period_label = period_labels.get(window_start, window_start)
            row_df = station_df[station_df["window_start"] == window_start]

            monthly_vals = []
            for p in param_sets:
                prow = row_df[row_df["param_set"] == p]
                monthly_vals.append("---" if prow.empty else fmt(prow["kge_monthly"].values[0]))

            # first period: print station name normally
            # subsequent periods: grey it out so it reads as a continuation
            if i == 0:
                station_cell = escaped_station
            else:
                station_cell = r"\textcolor{gray}{" + escaped_station + "}"

            row = " & ".join(
                [station_cell, period_label] + monthly_vals
            ) + r" \\"
            lines.append(row)

        lines.append(r"\addlinespace")

    lines += [
        r"\end{longtable}",
    ]

    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"Written to {output_path}")

In [16]:
df_to_latex_kge_monthly(
    csv_path=OUTPUT_PATH,
    output_path=PROJECT_ROOT / "outputs/validation_kge_table_monthly.tex",
)

Written to /home/avandervee3/aral_sea_full_project/outputs/validation_kge_table_monthly.tex


In [17]:
MAIN_TEXT_STATIONS = [
    ("Chatly",       "Amu Darya"),
    ("Kerki",        "Amu Darya"),
    ("Kazalinsk",    "Syr Darya"),
    ("Karaozek",     "Syr Darya"),
    ("Tyumen-aryk",  "Syr Darya"),
]

def df_to_latex_kge_monthly_main(
    csv_path: str | Path,
    output_path: str | Path,
    stations: list[tuple[str, str]],
    caption: str = "Monthly KGE scores for selected stations for calibration (1950--1954) and validation (1955--1959) periods.",
    label: str = "tab:kge_main",
):
    df = pd.read_csv(csv_path)

    def parse_param_set(experiment):
        for part in str(experiment).split("__"):
            if part.startswith("parameter_set="):
                return part.replace("parameter_set=", "")
        return experiment

    df["param_set"] = df["experiment"].apply(parse_param_set)

    param_labels = {
        "uncalibrated":       "Uncalibrated",
        "overall_calibrated": "Overall cal.",
        "amu_calibrated":     "Amu Darya cal.",
        "syr_calibrated":     "Syr Darya cal.",
    }

    param_sets = list(dict.fromkeys(df["param_set"].tolist()))

    period_labels = {}
    sorted_windows = sorted(df["window_start"].unique())
    period_labels[sorted_windows[0]] = "Calibration"
    for w in sorted_windows[1:]:
        period_labels[w] = "Validation"

    station_names = [s for s, _ in stations]
    river_lookup  = {s: r for s, r in stations}

    def fmt(val):
        if pd.isna(val):
            return "---"
        return f"${val:.2f}$"

    n_params   = len(param_sets)
    col_spec   = "ll" + "r" * n_params
    col_headers = " & ".join(param_labels.get(p, p) for p in param_sets)
    total_cols  = 2 + n_params

    lines = []
    lines += [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\begin{tabular}{" + col_spec + "}",
        r"\toprule",
        r"Station & Period & " + col_headers + r" \\",
        r"\midrule",
    ]

    for station in station_names:
        station_df = df[df["station_name"] == station]
        if station_df.empty:
            continue

        periods  = sorted(station_df["window_start"].unique())
        n_periods = len(periods)
        river    = river_lookup.get(station, "")
        escaped  = station.replace(".", r".\ ")

        # station name + river name stacked using \shortstack
        station_cell = (
            r"\multirow{" + str(n_periods) + r"}{*}{"
            r"\shortstack[l]{"
            + escaped + r"\\" +
            r"{\footnotesize \textit{" + river + r"}}"
            r"}}"
        )

        for i, window_start in enumerate(periods):
            period_label = period_labels.get(window_start, window_start)
            row_df = station_df[station_df["window_start"] == window_start]

            monthly_vals = []
            for p in param_sets:
                prow = row_df[row_df["param_set"] == p]
                monthly_vals.append("---" if prow.empty else fmt(prow["kge_monthly"].values[0]))

            first_cell = station_cell if i == 0 else ""
            row = " & ".join([first_cell, period_label] + monthly_vals) + r" \\"
            lines.append(row)

        lines.append(r"\addlinespace")

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"Written to {output_path}")


df_to_latex_kge_monthly_main(
    csv_path=OUTPUT_PATH,
    output_path=PROJECT_ROOT / "outputs/validation_kge_table_main.tex",
    stations=MAIN_TEXT_STATIONS,
)

Written to /home/avandervee3/aral_sea_full_project/outputs/validation_kge_table_main.tex
